In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

from var import DATA_OUT
from scintill_ai.preprocess import get_time_filtering_and_features

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
from neuralforecast.losses.pytorch import RMSE

In [ ]:
def rrmse(y_true, y_pred, y_train):
    rmse_test = rmse(y_true, y_pred)
    mean_train = np.mean(y_train)
    return rmse_test / mean_train

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_pred - y_true) ** 2))

In [ ]:
# Load pre-processed dataset
df = pd.read_parquet(Path(DATA_OUT, 'df.parquet'), engine='pyarrow').drop(
    columns=['s4_max']
)

# Filter by time window (from 20:00 to 05:59 UTC) and create temporal features (lag and MAVs)
df = get_time_filtering_and_features(
    df, hour_start=20, hour_stop=6, ema_cols=None, lag_cols=None,
)

# Nixtla-specific pre-processing
df = df.reset_index(names='ds').rename(columns={'s4_mean': 'y'})
df['unique_id'] = 1

In [ ]:
TRAIN_START, TRAIN_STOP = '2021-12-01', '2023-01-31'
TRAIN_START_idx, TRAIN_STOP_idx = df[df['ds'].eq(TRAIN_START)].index[0], df[df['ds'].eq(TRAIN_STOP)].index[0]

CALIB_START, CALIB_STOP = '2023-03-01', '2023-03-17'
CALIB_START_idx, CALIB_STOP_idx = df[df['ds'].eq(CALIB_START)].index[0], df[df['ds'].eq(CALIB_STOP)].index[0]

TEST_START, TEST_STOP = '2023-03-18', '2023-03-31'
TEST_START_idx, TEST_STOP_idx = df[df['ds'].eq(TEST_START)].index[0], df[df['ds'].eq(TEST_STOP)].index[0]

In [ ]:
nbeats_cols = [
    'ds',
    'unique_id',
    'y',
]

y_train, y_calib, y_test = (
    df.loc[TRAIN_START_idx:TRAIN_STOP_idx, nbeats_cols].copy().fillna(0),
    df.loc[CALIB_START_idx:CALIB_STOP_idx, nbeats_cols].copy().fillna(0),
    df.loc[TEST_START_idx:TEST_STOP_idx, nbeats_cols].copy().fillna(0),
)

y_train = pd.concat([y_train, y_calib])

In [ ]:
# tscv = TimeSeriesSplit(n_splits=3)

# model = CatBoostRegressor(
#     loss_function="RMSE",
#     cat_features=['s4_mean_variation_online'],
#     random_seed=42,
#     thread_count=-1,
#     bootstrap_type="Bernoulli",
#     sampling_frequency='PerTree',
#     od_type="Iter",
#     od_wait=300,
#     use_best_model=True,
#     subsample=0.9,
#     colsample_bylevel=0.9,
#     has_time=True,
# )

# param_grid = {
#     "iterations": [500, 1_000, 2_000],
#     "max_depth": [4, 6, 8],
#     "l2_leaf_reg": [10, 30],
#     "min_data_in_leaf": [50, 100],
# }

# grid_search = GridSearchCV(
#     estimator=model,
#     param_grid=param_grid,
#     cv=tscv,
#     scoring="neg_root_mean_squared_error",
#     n_jobs=-1,
#     verbose=1,
# )

# grid_search.fit(X_train, y_train, eval_set=(X_test, y_test))

# grid_search.best_params_
# # {'iterations': 500, 'l2_leaf_reg': 30, 'max_depth': 4, 'min_data_in_leaf': 50}

In [ ]:
quantile_mapping = {
    95: [0.025, 0.975],
    90: [0.05, 0.95],
    80: [0.10, 0.90],
}

In [ ]:
quantile_str = str(quantile_mapping[95]).replace('[','').replace(']','')

[params](https://catboost.ai/docs/en/references/training-parameters/)

In [ ]:
cb_multiquantile = CatBoostRegressor(
    loss_function=f'MultiQuantile:alpha={quantile_str}',
    thread_count=-1,
    bootstrap_type="Bernoulli",
    sampling_frequency='PerTree',
    iterations=3_000,
    od_type='Iter',
    od_wait=300,
    # use_best_model=True,
    max_depth=10,
    subsample=0.9,
    colsample_bylevel=0.9,
    min_data_in_leaf=50,
    verbose=1,
    random_seed=42,
)

cb_multiquantile.fit(X_train, y_train)

In [ ]:
y_pred_calib = cb_multiquantile.predict(X_calib)
y_pred_test = cb_multiquantile.predict(X_test)

In [ ]:
level = 95
q_low = quantile_mapping[level][0]
q_hig = quantile_mapping[level][1]

print(f'level : {level} interval : [{q_low} , {q_hig}]\n')

# Keep only predicted Quantiles for the calibration set related to the current level
quantile_regression_calibration_intervals = np.zeros([len(X_calib), 2])
quantile_regression_calibration_intervals[:, 0] = y_pred_calib[:, 0]
quantile_regression_calibration_intervals[:, 1] = y_pred_calib[:, 1]

# Compute Non-Conformity Measures on the Calibration set (How predicted Quantiles relate to the True value)
non_conformity_scores = np.max(
    [
        quantile_regression_calibration_intervals[:, 0] - y_calib,
        y_calib - quantile_regression_calibration_intervals[:, 1],
    ],
    axis=0,
)

# Sort Non-Conformity Measures
non_conformity_scores = np.sort(non_conformity_scores)[::-1]

# Compute the Quantile based on Non-Conformity Measures distribution with level as threshold
emperical_quantile = (level/100) * (1 + (1 / len(y_calib)))
correction_factor = np.quantile(non_conformity_scores, emperical_quantile, method="higher")

# # Plot the non_conformity_scores distribution
# plt.hist(non_conformity_scores, bins='auto', color='magenta')
# plt.axvline(correction_factor, color='black', linestyle='dashed', linewidth=1, label='quantile')

# plt.legend()
# plt.xlabel('Calibration Error')
# plt.ylabel('Frequency')
# plt.title('Histogram of Calibration Errors')
# plt.show()

# Keep only predicted Quantiles for the test set related to the current level
quantile_regression_prediction_intervals = np.zeros([len(X_test), 2])
quantile_regression_prediction_intervals[:, 0] = y_pred_test[:, 0]
quantile_regression_prediction_intervals[:, 1] = y_pred_test[:, 1]

# Apply Correction factor on test set
correction_factor_test = np.ones([len(y_test), 2])
correction_factor_test[:, 0] *= correction_factor
correction_factor_test[:, 1] *= correction_factor

y_pred_test[:,0] = quantile_regression_prediction_intervals[:, 0] - correction_factor_test[:, 0]
y_pred_test[:,1] = quantile_regression_prediction_intervals[:, 1] + correction_factor_test[:, 1]

In [ ]:
print(f'level : {level} interval : [{q_low} , {q_hig}]\n')

hrs_start = 23
hrs_stop = 26

plt.figure(figsize=(20, 10))
plt.plot(y_test.iloc[hrs_start*60:hrs_stop*60].values, color='tab:blue')

plt.fill_between(
    x=np.arange(stop=(hrs_stop-hrs_start)*60),
    y1=y_pred_test[hrs_start*60:hrs_stop*60,0],
    y2=y_pred_test[hrs_start*60:hrs_stop*60,1],
    color='gray',
    alpha=0.2,
    label='Prediction Interval',
)

plt.show()

In [ ]:
df_eval = pd.DataFrame()

In [ ]:
df_eval['y_test'] = y_test
df_eval['y_pred_low'] = y_pred_test[:,0].clip(0)
df_eval['y_pred_hig'] = y_pred_test[:,1]
df_eval['is_covered'] = df_eval['y_test'].ge(df_eval['y_pred_low']) & df_eval['y_test'].le(df_eval['y_pred_hig'])

In [ ]:
df_eval.to_parquet(Path(DATA_OUT, 'eval_cqr_manina.parquet'), engine='pyarrow')

In [ ]:
df_eval['is_covered'].sum() / df_eval.shape[0]

In [ ]:
df_eval['y_test'].le(df_eval['y_pred_hig']).sum() / df_eval.shape[0]

In [ ]:
cv = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

In [ ]:
# cb = CatBoostRegressor(
#     loss_function='RMSE',
#     thread_count=-1,
#     bootstrap_type="Bernoulli",
#     sampling_frequency='PerTree',
#     iterations=3_000,
#     od_type='Iter',
#     od_wait=300,
#     max_depth=10,
#     subsample=0.9,
#     colsample_bylevel=0.9,
#     min_data_in_leaf=50,
#     verbose=1,
#     random_seed=42,
# )

cb = CatBoostRegressor(
    loss_function='RMSE',
    **cb_params,
)

In [ ]:
aci_res = aci_ts_regressor_predict(
    model=cb,
    cv=cv,
    train_data=(
        pd.concat([X_train, X_calib]),
        pd.concat([y_train, y_calib]),
    ),
    test_data=(X_test, y_test),
    update_calibration=True,
    gamma=0.03,
    forecast_horizon=2,
    alpha_list=[1 - 0.95],
)

In [ ]:
df_eval = pd.DataFrame({
    'y_test': y_test,
    'y_pred_low': aci_res[0]['y_pis'][:, 0, 0],
    'y_pred_hig': aci_res[0]['y_pis'][:, 1, 0],
})

df_eval['is_covered'] = df_eval['y_test'].ge(df_eval['y_pred_low']) & df_eval['y_test'].le(df_eval['y_pred_hig'])

In [ ]:
# df_eval.to_parquet(Path(DATA_OUT, 'eval_aci.parquet'), engine='pyarrow')

In [ ]:
# df_eval = pd.read_parquet(Path(DATA_OUT, 'eval_aci.parquet'), engine='pyarrow')
df_eval['width'] = df_eval['y_pred_hig'] - df_eval['y_pred_low']
df_eval['width_category'] = pd.cut(df_eval['width'], bins=9)
df_eval['y_test_category'] = pd.cut(df_eval['y_test'], bins=9)
df_eval['is_covered'] = df_eval['y_test'].ge(df_eval['y_pred_low']) & df_eval['y_test'].le(df_eval['y_pred_hig'])

In [ ]:
df_agg_wc = df_eval.groupby('width_category', observed=False).agg(
    n_samples=('width','count'),
    n_covered=('is_covered','sum'),
    mean_y_test=('y_test', 'mean'),
)

df_agg_wc['perc_covered'] = np.round(
    100 * df_agg_wc['n_covered'].div(df_agg_wc['n_samples']),
    1,
)

In [ ]:
df_agg_wc

In [ ]:
df_agg_yc = df_eval.groupby('y_test_category', observed=False).agg(
    n_samples=('width','count'),
    n_covered=('is_covered','sum'),
    mean_width=('width', 'mean'),
)

df_agg_yc['perc_covered'] = np.round(
    100 * df_agg_yc['n_covered'].div(df_agg_yc['n_samples']),
    1,
)

In [ ]:
df_agg_yc

In [ ]:
def rrmse(y_true, y_pred, digit=3):
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mean_true = np.mean(y_true)
    return np.round(rmse / mean_true, digit)

def rmse(y_true, y_pred, digit=3):
    return np.round(np.sqrt(np.mean((y_pred - y_true) ** 2)), digit)

In [ ]:
print(
    f"RMSE: {rmse(y_true=y_test.values, y_pred=aci_res[0]['y_pred'])} | RRMSE: {rrmse(y_true=y_test.values, y_pred=aci_res[0]['y_pred'])}"
)

In [ ]:
aci_res_loaded = aci_res[0]

In [ ]:
df_eval = pd.DataFrame({
    'y_pred': aci_res_loaded['y_pred'],
    'y_pi_low': aci_res_loaded['y_pis'][:, 0, 0],
    'y_pi_upp': aci_res_loaded['y_pis'][:, 1, 0],
    'y_true': y_test,
})

df_eval['is_scint'] = df_eval['y_true'].ge(LW_S4_THRESHOLD)
df_eval['is_scint_covered'] = df_eval['is_scint'] & (
    df_eval['y_pi_low'].le(df_eval['y_true'])) & (
    df_eval['y_true'].le(df_eval['y_pi_upp'])
)
df_eval['is_scint_covered_upward'] = df_eval['is_scint'] & (
    df_eval['y_true'].le(df_eval['y_pi_upp'])
)

In [ ]:
perc_scint = df_eval['is_scint'].eq(True).sum() / df_eval.shape[0]
perc_scint_covered = df_eval['is_scint_covered'].eq(True).sum() / df_eval['is_scint'].eq(True).sum()
perc_scint_covered_upward = df_eval['is_scint_covered_upward'].eq(True).sum() / df_eval['is_scint'].eq(True).sum()

In [ ]:
print(
    f'Scintillation happens {perc_scint:.1%} of the time. Scintillation is covered {perc_scint_covered:.1%} of the time (upward: {perc_scint_covered_upward:.1%})'
)

In [ ]:
plot_dict = aci_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, ls='-', label="Actual (test)", c="tab:orange")
ax.plot(
    y_test.index,
    plot_dict[0]["y_pred"],
    lw=1,
    ls=':',
    c="tab:blue",
    label="Forecast",
)

for i, result_ in enumerate(plot_dict):
    y_pis = result_["y_pis"]
    color = plt.cm.Blues(1 - i/len(plot_dict))
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.3,
        color=color,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('CatBoost + ACI', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y %H:%M'))
ax.yaxis.grid(True, color='k', linewidth=0.2, alpha=0.4)
[ax.spines[s].set_visible(False) for s in ax.spines]
ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
# ax.set_xlim(y_test.index[0], y_test.index[-1])
ax.set_xlim(y_test.loc['2023-03-19 23'].index[0], y_test.loc['2023-03-20 00'].index[-1])
# ax.set_ylim(0, 0.85)

# plt.savefig('aci_.png', dpi=800, bbox_inches='tight')
plt.show()

## [NBEATS](https://nixtlaverse.nixtla.io/neuralforecast/models.nbeats.html)

In [ ]:
nbeats = NBEATS(
    h=y_test.shape[0],
    input_size=5,
    loss=RMSE(),
    stack_types = ['identity', 'trend', 'seasonality'],
    max_steps=10,
    # val_check_steps=3,
    # early_stop_patience_steps=2,
    random_seed=42,
)

In [ ]:
fcst = NeuralForecast(
    models=[nbeats],
    freq='min',
)

In [ ]:
fcst.fit(df=y_train)

In [ ]:
forecasts = fcst.predict(df=y_test)

In [ ]:
df_eval = forecasts[
    ['ds', 'NBEATS']
].join(
    y_test.reset_index(drop=True)['y']
).set_index('ds')

In [ ]:
rmse(df_eval['y'], df_eval['NBEATS'])

In [ ]:
rrmse(df_eval['y'], df_eval['NBEATS'], y_calib['y'])

In [ ]:
df_eval

In [ ]:
plt.plot(df_eval.loc['2023-03-31','NBEATS'])

In [ ]:
# df_eval = pd.read_parquet(Path(DATA_OUT, 'eval_cqr_ct.parquet'), engine='pyarrow')

In [ ]:
# df_agg_variation = df_eval.join(
#     df.loc[TEST_START:TEST_STOP, 's4_mean_variation_offline'],
# ).groupby(
#     by='s4_mean_variation_offline',
# ).agg(
#     n_covered=('is_covered', 'sum'),
#     n_samples=('is_covered', 'count'),
#     mean_y_test=('y_test', 'mean'),
#     median_y_test=('y_test', 'median'),
# )

# df_agg_variation['perc_covered'] = df_agg_variation['n_covered'].div(df_agg_variation['n_samples'])

In [ ]:
# df_agg_variation

In [ ]:
# px.line(df_eval['width'])

In [ ]:
# px.histogram(df_eval['width'], nbins=10)

In [ ]:
# df_agg_wc = df_eval.groupby('width_category', observed=False).agg(
#     n_samples=('width','count'),
#     n_covered=('is_covered','sum'),
#     mean_y_test=('y_test', 'mean'),
# )

# df_agg_wc['perc_covered'] = np.round(
#     100 * df_agg_wc['n_covered'].div(df_agg_wc['n_samples']),
#     1,
# )

In [ ]:
df_agg_yc = df_eval.groupby('y_test_category', observed=False).agg(
    n_samples=('width','count'),
    n_covered=('is_covered','sum'),
    mean_width=('width', 'mean'),
)

df_agg_yc['perc_covered'] = np.round(
    100 * df_agg_yc['n_covered'].div(df_agg_yc['n_samples']),
    1,
)

In [ ]:
df_agg_yc

In [ ]:
# fig, ax = plt.subplots()

# quantile = 0.5
# ax.scatter(y_pred_test_quant[quantile], y_test)

# min_val = min(y_pred_test_quant[quantile].min(), y_test.min())
# max_val = max(y_pred_test_quant[quantile].max(), y_test.max())
# ax.plot([min_val, max_val], [min_val, max_val], 'r', linestyle='--')

# ax.set_xlabel("Predicted <S4>")
# ax.set_ylabel("True <S4>")

# plt.show()

In [ ]:
rolling_cov = []
window = 180

for i in range(window, y_test.shape[0], 1):
    rolling_cov.append(
        (
            (y_pred_test_quant[0.025].values[i-window:i] <= y_test[i-window:i]) &
            (y_test[i-window:i] <= y_pred_test_quant[0.975].values[i-window:i])
        ).sum() / window
    )

In [ ]:
plt.figure(figsize=(10, 5))
plt.ylabel(f"Rolling coverage [{window} minutes]")

plt.plot(
    y_test[window:].index,
    rolling_cov,
    linestyle='--', color='tab:blue', alpha=0.8
)

marg_cov = df_eval['is_covered'].sum() / df_eval.shape[0]
plt.plot(
    y_test[window:].index,
    y_test[window:].shape[0] * [marg_cov],
    linestyle='-', color='tab:orange', alpha=0.8,
)

plt.show()

In [ ]:
dt_1, dt_2 = '2023-03-18 00', '2023-03-18 03'

plt.figure(figsize=(20, 10))
plt.plot(y_test.loc[dt_1:dt_2], color='tab:blue')

plt.fill_between(
    x=y_pred_test_quant.loc[dt_1:dt_2].index,
    y1=y_pred_test_quant.loc[dt_1:dt_2,0.975],
    y2=y_pred_test_quant.loc[dt_1:dt_2,0.025],
    color='gray',
    alpha=0.2,
    label='Prediction Interval',
)

plt.show()